# Lecture 3 — Clutter is Your Enemy + Line Charts: Class Exercise

**Dataset:** CO2 Emissions by Country 2000–2022 — [Our World in Data](https://ourworldindata.org/co2-emissions)

**Rules:** no spaghetti (grey everything, highlight one), remove clutter, label directly, insight title.

## 0. Setup: get the data
Runs once. If `../data/co2_emissions.csv` already exists, it is reused.

In [1]:
import os
import pandas as pd
import plotly.express as px

DATA_PATH = '../data/co2_emissions.csv'

if not os.path.exists(DATA_PATH):
    url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
    raw = pd.read_csv(url)
    # keep real countries only (drop world/continents/income groups), 2000-2022
    d = raw[raw['iso_code'].notna() & ~raw['iso_code'].astype(str).str.startswith('OWID')]
    d = d[(d['year'] >= 2000) & (d['year'] <= 2022)][['country', 'year', 'co2']].dropna()
    d = d.rename(columns={'country': 'Country', 'year': 'Year', 'co2': 'CO2_Mt'})
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    d.to_csv(DATA_PATH, index=False)
    print('Downloaded and saved', DATA_PATH)

Downloaded and saved ../data/co2_emissions.csv


## 1. Load and inspect

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
df.head()

Loaded: 4945 rows | Countries: 215 | Years: 2000-2022


,Country,Year,CO2_Mt
0,Afghanistan,2000,1.047
1,Afghanistan,2001,1.069
2,Afghanistan,2002,1.341
3,Afghanistan,2003,1.560
4,Afghanistan,2004,1.237


## 2. Example 1 — Single line: before and after decluttering

In [3]:
usa = df.loc[df['Country'] == 'United States']

# BEFORE: default styling
fig = px.line(usa, x='Year', y='CO2_Mt', title='CO2 Emissions over time', markers=True)
fig.show()

In [4]:
# AFTER: clutter removed
first, last = usa['Year'].min(), usa['Year'].max()
chg = (usa.loc[usa['Year'] == last, 'CO2_Mt'].values[0] / usa.loc[usa['Year'] == first, 'CO2_Mt'].values[0] - 1) * 100

fig = px.line(usa, x='Year', y='CO2_Mt',
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''},
              color_discrete_sequence=['#2E75B6'],
              template='plotly_white',      # Closure: no chart border needed
              markers=False,                # Connection: the line already links the points
              height=400)

fig.update_layout(
    title=f'US CO2 emissions fell {abs(chg):.0f}% between {first} and {last}',
    font=dict(size=13),
    yaxis=dict(gridcolor='#EEEEEE'),        # Continuity: light gridlines
    xaxis=dict(showgrid=False),
)
fig.update_traces(line=dict(width=2.5))
fig.show()

## 3. Example 2 — Multi-series: the spaghetti problem

In [5]:
# BEFORE: 200+ lines
fig = px.line(df, x='Year', y='CO2_Mt', color='Country',
              title='CO2 Emissions by Country', height=500)
fig.show()

In [6]:
# AFTER: grey context, one highlighted country, direct label
highlight = 'China'
first, last = df['Year'].min(), df['Year'].max()
start = df.loc[(df['Country'] == highlight) & (df['Year'] == first), 'CO2_Mt'].values[0]
end   = df.loc[(df['Country'] == highlight) & (df['Year'] == last),  'CO2_Mt'].values[0]

color_map = {c: 'crimson' if c == highlight else 'lightgrey' for c in df['Country'].unique()}

fig = px.line(df, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''},
              template='plotly_white', height=400)

fig.update_traces(line=dict(width=1.5), showlegend=False)
fig.update_traces(line=dict(width=2.5), selector=dict(name=highlight))

fig.add_annotation(x=last, y=end, text=highlight, showarrow=False,
                   xanchor='left', xshift=5,
                   font=dict(color='crimson', size=12, family='Arial', weight='bold'))

fig.update_layout(
    title=f"{highlight}'s CO2 emissions grew {end/start:.1f}x since {first} — no other country comes close",
    margin=dict(r=100),
    yaxis=dict(gridcolor='#EEEEEE'),
    xaxis=dict(showgrid=False),
)
fig.show()

## 4. Class Exercise 💪

Pick a **different** country, grey out everything else, label directly, and write an insight title with a number from the data.

In [7]:
highlight = 'India'   # change to any country in df['Country']

first, last = df['Year'].min(), df['Year'].max()
start = df.loc[(df['Country'] == highlight) & (df['Year'] == first), 'CO2_Mt'].values[0]
end   = df.loc[(df['Country'] == highlight) & (df['Year'] == last),  'CO2_Mt'].values[0]

color_map = {c: '#C00000' if c == highlight else '#D9D9D9' for c in df['Country'].unique()}

fig = px.line(df, x='Year', y='CO2_Mt', color='Country',
              color_discrete_map=color_map,
              labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''},
              template='plotly_white', height=450)

fig.update_traces(line=dict(width=1.2), showlegend=False)
fig.update_traces(line=dict(width=3), selector=dict(name=highlight))

# Direct label on the highlighted line
fig.add_annotation(x=last, y=end, text=highlight, showarrow=False,
                   xanchor='left', xshift=6,
                   font=dict(color='#C00000', size=13, family='Arial', weight='bold'))

# One quiet label for the grey lines (second-largest emitter in the last year, since the top one is usually China)
ctx = df.loc[(df['Country'] != highlight) & (df['Year'] == last)].nlargest(2, 'CO2_Mt').iloc[[-1]]
fig.add_annotation(x=last, y=ctx['CO2_Mt'].values[0], text='Other countries', showarrow=False,
                   xanchor='left', xshift=6, font=dict(color='#A6A6A6', size=12, family='Arial'))

fig.update_layout(
    title=f"{highlight}'s CO2 emissions grew {end/start:.1f}x between {first} and {last}",
    font=dict(size=13),
    margin=dict(r=130),
    yaxis=dict(gridcolor='#EEEEEE', zeroline=False),
    xaxis=dict(showgrid=False),
)
fig.show()